In [38]:
import tensorflow as tf 
tf.config.list_physical_devices("GPU")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [39]:
import os

# Define paths
video1_path = 'video/Falcon-heavy-Starman(720p).mp4'
video2_path = 'video/Falcon-heavy-Starman(lowQ).mp4' # You will have to transform a good quality video iin something resembling old with python

# Create the folders
output_folder1 = "video/frames_720p"
output_folder2 = "video/frames_lowQ"

os.makedirs(output_folder1, exist_ok=True)
os.makedirs(output_folder2, exist_ok=True)

# Read videos
import cv2
cap1 = cv2.VideoCapture(video1_path)
cap2 = cv2.VideoCapture(video2_path)

# Get all frames
total_frames1 = int(cap1.get(cv2.CAP_PROP_FRAME_COUNT))
total_frames2 = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))

# Print the total frames 
print('Total frames 1: ', total_frames1)
print('Total frames 2: ', total_frames2)

Total frames 1:  2696
Total frames 2:  3372


In [40]:
# Preprocessing
# 
# Extract frames
for i in range(total_frames1):
    cap1.set(cv2.CAP_PROP_POS_FRAMES, i)
    ret, frame = cap1.read()
    if ret:
        cv2.imwrite(os.path.join(output_folder1, f"frame_{i}.jpg"), frame)
    # Uniform sampling from the video with more frames (lowQ) in case one of the videos has more frames 
    step_size = total_frames1/total_frames2
for i in range(total_frames2):
    frame_pos = int(i*step_size)
    cap2.set(cv2.CAP_PROP_POS_FRAMES, frame_pos)
    ret, frame = cap2.read()
    if ret:
        cv2.imwrite(os.path.join(output_folder2, f"frame_{i}.jpg"), frame)
            # 
# Release
cap1.release()
cap2.release()
cv2.destroyAllWindows()

In [41]:
# s = cv2.imread("video/frames_lowQ/frame_0.jpg")
# s.shape

In [42]:
# Path to the low quality frame
frame_path_lowQ = "video/frames_lowQ/frame_1992.jpg"

frame_lowQ = cv2.imread(frame_path_lowQ)
height, width, channels = frame_lowQ.shape
print(f"Frame 1992 size is: {height}x{width}x{channels}")

frame_path_720p = "video/frames_720p/frame_1992.jpg"

frame_720p = cv2.imread(frame_path_720p)
height, width, channels = frame_720p.shape
print(f"Frame 1992 size is: {height}x{width}x{channels}")


Frame 1992 size is: 288x352x3
Frame 1992 size is: 720x1280x3


In [ ]:
# Notice this is only because I executed the transdormation outside python, I won't need this for workshop 3
import matplotlib.pyplot as plt

frames_dir = "video/frames_lowQ"

# for filename in os.listdir(frames_dir):
    # frame_path = os.path.join(frames_dir, filename)
    # frame = cv2.imread(frame_path)
    # Remove 45 pixels from top and bottom
    # height, width, _ = frame.shape
    # cropped_frame = frame[45:height-45, :]
    # Overwrite the original file
    # cv2.imwrite(frame_path, cropped_frame) # Comment this line if already run once
    # print(cropped_frame.shape, frame.shape)
    # plt.imshow(cropped_frame)
    # plt.show()
    # plt.imshow(frame_720p)
    # break

In [44]:
# Unoptimized code

import numpy as np

# Load the images 
def load_images(lowQ_dir, highQ_dir):
    """
    This function loads our images at once
    
    In order to this function to load all the images we explore both directories and then we load the images by appending them to a list
    and transforming them to a numpy array.
    
    Args:
        lowQ_dir (str): Where the low q frames are
        highQ_dir (str): Where the high q frames are
        
    Returns:
        (np.array): numpy array for the low q frames
        (np.array): numpy array for the high q frames
    """
    lowq_images = []
    highq_images = []
    for filename in sorted(os.listdir(lowQ_dir)):
        lowq_path = os.path.join(lowQ_dir, filename)
        highq_path = os.path.join(highQ_dir, filename)
        lowq_img = cv2.imread(lowq_path)
        highq_img = cv2.imread(highq_path)
        lowq_images.append(lowq_img)
        highq_images.append(highq_img)
    return np.array(lowq_images),np.array(highq_images) 

# Load the function
lowq_dir = "video/frames_lowQ"
highq_dir = "video/frames_720p"

lowq_images, highq_images = load_images(lowq_dir, highq_dir)

C:\Users\joral\AppData\Local\Temp\ipykernel_14612\2446539067.py:30: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(lowq_images),np.array(highq_images)


In [ ]:
# Tensorflow way (Optimized)
import tensorflow as tf

def create_dataset(lowq_dir, highq_dir, batch_size):
    """
    This function will help with the dynamic dataset
    
    This function does implement the generator for the images in which we will
    load to our model in the form of a tensor
    
    Args: 
        lowq_dir: directory in which i saved the lowq images
        highq_dir: directory in which i saved the highq images
        batch_size: size of the data in each batch
        
    Returns:
        dataset (tf.data.Dataset): the dynamic data
        steps_per_epoch (int): the number of steps per epoch
    """

    def load_image_pair(lowq_filepath, highq_filepath):
        """
        This function loads a image pair
        
        The function load a loeq image together with a higq image. we also normalize the image when loading
        
        Args: 
            lowq_filepath (str): directory in which i saved the lowq images
            highq_filepath (str): directory in which i saved the highq images
        
        Returns: 
            lowq_img (cv2.Image):
            highq_img (cv2.Image):
        """
        lowq_img = cv2.imread(lowq_filepath)
        highq_img = cv2.imread(highq_filepath)
        
        # Preprocess
        lowq_img = lowq_img/255
        highq_img = highq_img/255
        
        lowq_img = lowq_img.astype(np.float32)
        highq_img = highq_img.astype(np.float32)
        
        return lowq_img, highq_img
        
    def generator():
        """
        This function will return the images in a dynamic way
        
        Yield:
            (tuple): images we want to load
        """
        for lowq_filepath, highq_filepath in zip(lowq_filepaths_list, highq_filepaths_list):
            yield load_image_pair(lowq_filepath, highq_filepath)
            
    # Load a list of the filenames of the frames for each video
    lowq_filenames_list = sorted(os.listdir(lowq_dir))
    highq_filenames_list = sorted(os.listdir(highq_dir))
    
    # Convert to path
    lowq_filepaths_list = [os.path.join(lowq_dir, filename) for filename in lowq_filenames_list]
    highq_filepaths_list = [os.path.join(highq_dir, filename) for filename in highq_filenames_list]
    
    # Detect the shape from the first image pair
    lowq_img, highq_img = load_image_pair(lowq_filenames_list[0], highq_filepaths_list[0])
    lowq_image_shape, highq_image_shape = lowq_img.shape, highq_img.shape
    
    #Give the model what the model wants
    dataset = tf.data.Dataset.from_generator(generator, output_signature=(tf.TensorSpec(shape=lowq_image_shape, dtype = tf.float32),
                                             tf.TensorSpec(shape=highq_image_shape,dtype = tf.float32)))
    
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    # Calculate the number of steps per epoch
    assert(len(lowq_filepaths_list) == len(highq_filepaths_list))
    total_number_of_samples = len(lowq_filepaths_list)
    steps_per_epoch = total_number_of_samples // batch_size
    
    return dataset, steps_per_epoch

In [49]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, Resizing

def build_autoencoder(input_shape, output_shape= (720,1280,3)):
    """
    
    """
    inputs = Input(shape= input_shape)
    # 1st part  (downscaling)
    x = Conv2D(32, 3, strides=2, padding="same", activation="relu")(inputs)
    # x = MaxPooling2D((2,2), padding = "same")  # + upscale
    x = Conv2D(64, 3, strides=2, padding="same", activation="relu")(x)
    
    # 2nd part (upscaling)
    x = Conv2DTranspose(32, 3, strides = 2, padding = "same", activation="relu")(x)
    x = Conv2DTranspose(16, 3, strides = 2, padding = "same", activation="relu")(x)
    x = Conv2DTranspose(8, 3, strides = 2, padding = "same", activation="relu")(x)

    x = Resizing(720,1280, interpolation="bilinear")(x)
    
    outputs = Conv2D(3, 3, padding = 'same', activation = 'sigmoid')(x) # check tanh and others
    
    model = Model(inputs, outputs)
    
    return model

In [50]:
lowq_dir = "video/frames_lowQ"
highq_dir = "video/frames_720p"
dataset, steps_per_epoch = create_dataset(lowq_dir, highq_dir, batch_size=2) # use larger batch size on coding tests

input_shape = dataset.element_spec[0].shape[1:]

model = build_autoencoder(input_shape= input_shape)

model.compile(optimizer="adam", loss = "mse")
model.summary()
model.fit(dataset,epochs = 5, steps_per_epoch = steps_per_epoch, verbose=1)

TypeError: unsupported operand type(s) for /: 'NoneType' and 'int'

In [ ]:
# Evaluation

# import os, cv2

# Define paths
video_path = "video/moon_salute.mp4"
output_folder = "video/moon_salute"

# Create the output folder
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    
# Load the video
cap = cv2.VideoCapture(video_path)

# Get video properties
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Total frames: {frame_count}")

# Extract the frames
frame_number = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    # Save the frames
    output_path = os.path.join(output_folder, f"frame_{frame_number:04}.jpg")
    cv2.imwrite(output_path, frame)
    frame_number += 1
    
# Release video
cap.release()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# Define folders
input_folder = "video/moon_salute"
output_folder = "video/moon_salute_enhanced"

# Create output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    
# Video changes
def resize_with_padding(frame, target_height, target_width):
    """
    This function will resize the image passed to it based on a given tartet height and width while adjusting the aspect ratio in order
    to match the training video    

    Args:
        frame (cv2 image): original test frame
        target_height (int): target height
        target_width (int): target width

    Returns:
        padded (cv2 image): new test frame with the right dimensions
    """
    # Get original dimensions
    h, w = frame.shape[:2]
    aspect_ratio = w/h
    target_aspect_ratio = target_width/target_height
    
    if aspect_ratio > target_aspect_ratio:
        # Frame is woder so we will ad padding to the height
        new_width = target_width
        new_height = int(new_width/aspect_ratio)
        resized = cv2.resize(frame, (new_width, new_height), interpolation= cv2.INTER_AREA)
        top_pad = (target_height - new_height)//2
        bottom_pad = target_height - new_height - top_pad
        padded = cv2.copyMakeBorder(resized, top_pad, bottom_pad, 0,0, cv2.BORDER_CONSTANT, value = (0,0,0))
    else:
        # Frame is woder so we will ad padding to the height
        new_height = target_height
        new_width = int(new_height/aspect_ratio)
        resized = cv2.resize(frame, (new_width, new_height), interpolation= cv2.INTER_AREA)
        left_pad = (target_width - new_width)//2
        right_pad = target_width - new_width - left_pad
        padded = cv2.copyMakeBorder(resized, 0, 0, left_pad, right_pad, cv2.BORDER_CONSTANT, value = (0,0,0))
    return padded

